**TIP:** You can save your work offline by using `File -> Download`.
Later you can upload the notebook back into the workspace.

## Setup

In [ ]:
# SETUP CELL
%pip -q install numpy sympy matplotlib ipywidgets pandas plotly anywidget scipy

In [ ]:
# SETUP CELL
import src_path
from gu_toolkit import *
from helpers.Fourier_01_helper import create_mystery_function
from helpers.Fourier_02_helper import MaxDistanceCard, AvgDistanceCard

# Recovering a mystery sine series via `NIntegrate`

In `Fourier_01` the mystery function had the form
$$
F_{\text{myst}}(x)=\alpha_1\sin(2\pi x)+\alpha_2\sin(2\pi 2x)+\cdots+\alpha_N\sin(2\pi N x)
$$
with random coefficients between `-1` and `1`.

In this notebook you will recover those coefficients numerically, copy them into sliders, and then use one parameter `N` to turn on the partial sums.

## Generate a mystery function

In [ ]:
Nmyst = 9
Fmyst = create_mystery_function(Nmyst)

display(
    Latex(
        r"$F_{\text{myst}}(x)$ is a random sine sum with "
        + f"{Nmyst}"
        + r" hidden modes."
    )
)

In [ ]:
fig_mystery = Figure(x_range=(-1 / 2, 1 / 2))
fig_mystery.title = r"The mystery function $F_{\mathrm{myst}}(x)$"
display(fig_mystery)

with fig_mystery:
    plot(Fmyst(x), x, id="F_myst", label=r"$F_{\text{myst}}(x)$")

## Compute the Fourier coefficients

On the interval $[-\tfrac12,\tfrac12]$, compute
$$
c_n = 2\int_{-1/2}^{1/2} F_{\text{myst}}(x)\sin(2\pi n x)\,dx.
$$
Because the mystery function is already a finite sine sum, the nonzero values should reveal the hidden coefficients.

In [ ]:
Nmax = Nmyst

FTb = np.zeros(Nmax + 1)
for n in range(1, Nmax + 1):
    FTb[n] = NIntegrate(
        2 * Fmyst(x) * sin(2 * pi * n * x),
        (x, -1 / 2, 1 / 2),
    )



coeff_table = pd.DataFrame(
    {
        "n": np.arange(1, Nmyst + 1),
        "target value for b_n": coeff_target[1:Nmyst+1],
        "|b_n|": np.abs(coeff_target[1:Nmyst+1]),
    }
)
display(coeff_table.round(1))

## Part 1. Use the table to set the slider parameters

Use the table above as the answer key for the model
$$
M(x)=a_1\sin(2\pi x)+a_2\sin(2\pi 2x)+\cdots+a_{N_{\max}}\sin(2\pi N_{\max}x).
$$
Copy the computed values into the sliders `a_1, a_2, \ldots, a_{N_{\max}}`. The coefficients after the hidden modes should stay at `0`.

In [ ]:
slider_limit = 1.5
a_symbols = [a[n] for n in range(1, Nmax + 1)]

model = 0
for n in range(1, Nmyst + 1):
    model += a[n] * sin(2 * pi * n * x)

fig = Figure(x_range=(-1 / 2, 1 / 2))
fig.show()

with fig:
    set_title(r"Mystery function and the slider-controlled sine model")
    plot(Fmyst(x), x, id="F_myst", label=r"$F_{\text{myst}}(x)$")
    plot(model, x, id="model", label=r"$M(x)$")
        

## Part 2. Add one parameter-controlled partial sum

Now build
$$
S_N(x)=\sum_{n=1}^{N_{\max}} b_n \cdot \sin(2\pi n x)\cdot \left(\begin{cases}1& \text{ if }n\leq N\\ 0 &\text{ otherwise}\end{cases}\right).
$$
Then use the slider `N` to switch on the modes one by one. Once `N` reaches the hidden number of modes, the partial sum should already match the mystery function.

In [ ]:
N = symbols("N")

S_N = 0
for n in range(1, Nmax + 1):
    S_N += FTb[n] * Piecewise((1, n <= N), (0, True)) * sin(2 * pi * n * x)

with fig:
    parameter(N, min=0, max=Nmax, step=1, value=0)
    plot(S_N, x, id="S_N", label=r"$S_N(x)$")

fig.info(MaxDistanceCard(x, Fmyst(x), S_N), id="myst_minus_SN_max")
fig.info(AvgDistanceCard(x, Fmyst(x), S_N), id="myst_minus_SN_avg")

## Explore

- Which entries in the coefficient table are nonzero?
- After you copy the table values into the sliders, at which value of `N` does `S_N` become exact?
- What happens if you change the random seed or change `Nmyst`?
- If you increase `Nmax`, do the new coefficients stay close to `0`?